# GLM Algorithm

In [ ]:
#
# A GLM (Generalized Linear Model) lets you model relationships between variables using 
# linear predictors and flexible distributions, enabling regression and classification 
# tasks. I suggest to have a brief look at the swimlane diagram in the security and 
# privacy section (../security-and-privacy/Security & Privacy GLM.pdf) to have a good 
# overview of the different steps in the algorithm. From the diagram you can see that 
# this is a two (federated-)step itterative algorithm:
#
# 1. Call `compute_local_betas`
# 2. Call `compute_local_deviance`
#
# These two steps are repeated until the algorithm converges. And then there is the 
# central part responsible for the aggregation of the results. The central part of the 
# algorithm (the main call) will return the GLM estimate for the entire federated dataset. 
#
# 1. Create a new vantage6 task to execute the *glm* method (central 
#    part). This central part will start the tasks `compute_local_betas` and
#    `compute_local_deviance` (as you can see in the swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* GLM estimate from the central part (the main call)
#

In [ ]:
import base64
import json
import requests

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS

In [ ]:
ORGANIZATION_IDS = [1]

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 4
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 3

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "glm"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [99, 100]

In [ ]:
payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": [
        {
            "id": ORGANIZATION_IDS[0], # Central task
            "arguments": base64.b64encode(
                json.dumps(
                    {
                        "family": "poisson",
                        "predictor_variables": ["age", "sex"],
                        "outcome_variable": "new_surv",
                        "organizations_to_include": ORGANIZATION_IDS,
                    }
                ).encode("UTF-8")
            ).decode("UTF-8")
        }
    ],
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

In [ ]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

In [ ]:
# Get the results of the (central) task, thus the *global* summary statistics.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

result_global = json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

results = result_global['cohorts']
cohorts = list(results.keys())
predictors = [k for k in result_global['cohorts'][cohorts[0]]['coefficients']['beta'].keys() if k != 'Intercept']

# This set of four figures is a complete explanation of:
# * What the predictors do
# * How strong the effects are
# * Whether the model is useful
# * Whether the outputs are trustworthy

# ============================================================
# GROUPED COEFFICIENT / FOREST PLOT
# ============================================================

def plot_coefficients():
    '''
    Coefficient (Forest) Plot
    "Which predictors have an effect, and in which direction?"
    This plot shows how each predictor (e.g. age, sex) influences the outcome (direction + size + uncertainty of effects), separately for each cohort.

    How to read it:
    * Each point = estimated effect (β) of a predictor.
    * The bar = 95% confidence interval (the uncertainty around the estimate).
    * The vertical dashed line at 0 = “no effect”.

    Interpretation:
    * If the bar is entirely to the right of 0, the predictor increases the outcome.
    * If entirely to the left of 0, the predictor decreases the outcome.
    * If the bar crosses 0, the effect is uncertain → “no clear evidence”.
    '''

    fig, ax = plt.subplots(figsize=(7,4))
    y = np.arange(len(predictors))
    offset = 0.12

    for i, cohort in enumerate(cohorts):
        r = results[cohort]
        betas = np.array([r['coefficients']['beta'][p] for p in predictors])
        ses = np.array([r['coefficients']['std_error'][p] for p in predictors])
        ci_low = betas - 1.96 * ses
        ci_high = betas + 1.96 * ses
        ax.errorbar(
            betas, y + (offset if i else -offset),
            xerr=[betas - ci_low, ci_high - betas],
            fmt='o', capsize=4,
            label=cohort
        )

    ax.axvline(0, linestyle='--', color='grey')
    ax.set_yticks(y)
    ax.set_yticklabels(predictors)
    ax.set_title("Coefficients by cohort (95% CI)")
    ax.set_xlabel("Effect size (β)")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

plot_coefficients()


# ============================================================
# DEVIANCE REDUCTION BAR PLOT
# ============================================================

def plot_deviance():
    '''
    Deviance Reduction Plot
    "Does adding the predictors improve the model?"
    
    Null deviance = model with no predictors (just an intercept).
    Model deviance = model with your predictors (e.g. age, sex).

    How to read it:
    * Two bars per cohort:
        Light grey = null model
        Darker grey = model with age and sex
    * The difference (Δ) shows how much the predictors improve the fit.

    Interpretation:
    * Big drop = predictors explain a lot of variation in the data.
    * Small drop = predictors don't add much information.
    '''

    fig, ax = plt.subplots(figsize=(7,4))
    width = 0.35
    x = np.arange(len(cohorts))

    nulls = [results[c]['details']['null_deviance'] for c in cohorts]
    devs = [results[c]['details']['deviance'] for c in cohorts]

    ax.bar(x - width/2, nulls, width, label="Null deviance", color='#bbbbbb')
    ax.bar(x + width/2, devs, width, label="Model deviance", color='#888888')

    for i, (nd, md) in enumerate(zip(nulls, devs)):
        ax.text(i, min(nd,md), f"Δ={nd-md:.0f}", ha='left', va='bottom')

    ax.set_xticks(x)
    ax.set_xticklabels(cohorts)
    ax.set_ylabel("Deviance")
    ax.set_title("Deviance reduction")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

plot_deviance()


# ============================================================
# SUMMARY TABLE
# ============================================================

def plot_dataframe_table(results, cohorts, predictors):
    '''
    Summary Table
    "What are the actual numbers for each predictor?"
    
    How to read it:
    β = Size and direction of effect (positive or negative); how much the outcome changes when this predictor increases by 1 unit.
    SE = Uncertainty of β (lower = more certainty)
    z = How many standard errors β is away from 0
    p-value = Probability the effect is just noise (very small = strong evidence)

    '''
    rows = []
    for cohort in cohorts:
        r = results[cohort]
        for p in predictors:
            rows.append({
                "Cohort": cohort,
                "Predictor": p,
                "β": round(r['coefficients']['beta'][p], 4),
                "SE": round(r['coefficients']['std_error'][p], 4),
                "z": round(r['coefficients']['z_value'][p], 1),
                "p-value": "<1e-300" if r['coefficients']['p_value'][p] == 0 else f"{r['coefficients']['p_value'][p]:.1e}",
            })

    df = pd.DataFrame(rows)
    return df

print("SUMMARY OF KEY PARAMETERS")
df_summary = plot_dataframe_table(results, cohorts, predictors)
display(df_summary)


# ============================================================
# MODEL NOTES
# ============================================================

def model_notes_dataframe(results, results_meta):
    '''
    Model Notes
    This datframe displays meta‑information about each model.
    "Was the model fitted correctly, and how good was the fit?"
    "Did the model run successfully?"
    "Did the predictors help?"
    "How many data points were used?"

    How to read it:
    * Converged = The algorithm successfully found a solution → model is reliable
    * Dispersion = Should be ~1 for GLMs with Poisson/Gaussian; indicates model assumptions
    * Number of observations = Number of observations used
    * Number of variables = Number of predictors used
    * Null deviance = Error with no predictors
    * Model deviance =Error with predictors

    How to interpret:
    * Converged = True; The model is stable and trustworthy.
    * Model deviance < Null deviance; Predictors help explain the outcome.
    * Large drop between null and model deviance; Predictors are highly informative.
    '''

    rows = []

    for cohort, r in results.items():
        details = r["details"]
        rows.append({
            "Cohort": cohort,
            "Converged": details["converged"],
            "Dispersion": details["dispersion"],
            "Dispersion estimated?": details["is_dispersion_estimated"],
            "Number of observations": details["num_observations"],
            "Number of variables": details["num_variables"],
            "Null deviance": details["null_deviance"],
            "Model deviance": details["deviance"],
        })

    # Add global notes
    df = pd.DataFrame(rows)

    print("MODEL NOTES")
    print("Global iterations:", results_meta["iterations"])
    print("All models converged:", results_meta["all_converged"])

    return df

display(model_notes_dataframe(results, result_global['details']))
